# Step 1: Install libraries


In [ ]:
!pip install bertopic umap-learn hdbscan sentence-transformers pandas numpy matplotlib plotly wordcloud -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 3.4 MB/s eta 0:00:00


# Step 2: Import libraries


In [ ]:


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from wordcloud import WordCloud
import os
import glob
import warnings
warnings.filterwarnings('ignore')

# Step 3: Upload and unzip your file


In [ ]:
from google.colab import files
print("Please upload your w4_cop26_english.zip file:")
uploaded = files.upload()


Please upload your w4_cop26_english.zip file:


Saving w4_cop26_english.zip to w4_cop26_english.zip


In [ ]:
import zipfile
zip_file = [f for f in os.listdir() if f.endswith('.zip')][0]
with zipfile.ZipFile(zip_file, 'r') as zip_ref:
    zip_ref.extractall('/content/cop26')

print("Files extracted!")
!ls -la /content/cop26/ | head -20

Files extracted!
total 16
drwxr-xr-x 4 root root 4096 Apr  7 14:53 .
drwxr-xr-x 1 root root 4096 Apr  7 14:53 ..
drwxr-xr-x 2 root root 4096 Apr  7 14:53 cop26_english
drwxr-xr-x 3 root root 4096 Apr  7 14:53 __MACOSX


# Step 4: Load all country statement files


In [ ]:
documents = []
country_names = []
file_sizes = []

# Find the correct folder path
root_dir = '/content/cop26'


In [ ]:
# Navigate through folders (handles __MACOSX and nested folders)
for filepath in glob.glob(os.path.join(root_dir, '**/*.txt'), recursive=True):
    # Skip macOS system files
    if '__MACOSX' in filepath:
        continue

    # Get country name from filename (remove .txt)
    filename = os.path.basename(filepath)
    country_name = filename.replace('.txt', '').replace('_', ' ')

    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            content = f.read()
            if len(content.strip()) > 100:  # Only substantial documents
                documents.append(content)
                country_names.append(country_name)
                file_sizes.append(len(content))
    except:
        # Try different encoding if utf-8 fails
        try:
            with open(filepath, 'r', encoding='latin-1') as f:
                content = f.read()
                if len(content.strip()) > 100:
                    documents.append(content)
                    country_names.append(country_name)
                    file_sizes.append(len(content))
        except:
            print(f"Could not read: {filename}")

print(f"\n{'='*50}")
print(f"LOADED {len(documents)} COUNTRY STATEMENTS")
print(f"{'='*50}")
print(f"Sample countries: {', '.join(country_names[:10])}")
print(f"\nAverage document size: {np.mean(file_sizes):.0f} characters")
print(f"Total vocabulary: {len(' '.join(documents).split()):,} words")


LOADED 163 COUNTRY STATEMENTS
Sample countries: Holy See, NGO CGIAR, Mali, IO IUCN, Marshall Islands, Rwanda, Portugal, UNFCCC Executive Secretary, Bangladesh, Bhutan

Average document size: 4021 characters
Total vocabulary: 104,567 words


# Step 5: Preview one statement


In [ ]:
print(f"\n{'='*50}")
print(f"SAMPLE: {country_names[0]}")
print(f"{'='*50}")
print(documents[0][:800])
print("...")


SAMPLE: Holy See
Excellency,
Ladies and Gentlemen,
I have the honor of read an excerpt of the Message of Pope Francis, addressed to the President of this Conference.
We can achieve the goals set by the Paris Agreement only if we act in a coordinated and responsible way. Those goals are ambitious, and they can no longer be deferred. Today it is up to you to take the necessary decisions.
We are facing an epochal change that calls for commitment on the part of all, particularly those countries possessed of greater means. These countries need to take a leading role.
For its part, the Holy See has adopted a strategy of net-zero emissions operating on two levels: 1) the commitment of Vatican City State to achieve this goal by 2050; and 2) the commitment of the Holy See to promote education in integral ecology to
...


# Step 6: Run BERTopic


In [ ]:
print(f"\n{'='*50}")
print("RUNNING BERTOPIC PIPELINE")
print(f"{'='*50}")

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = embedding_model.encode(documents, show_progress_bar=True)

topic_model = BERTopic(
    embedding_model=embedding_model,
    min_topic_size=3,  # Smaller because we have only 163 docs
    verbose=True,
    calculate_probabilities=True
)

topics, probabilities = topic_model.fit_transform(documents)


RUNNING BERTOPIC PIPELINE


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

2026-04-07 14:57:09,689 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/6 [00:00<?, ?it/s]

2026-04-07 14:57:28,458 - BERTopic - Embedding - Completed ✓
2026-04-07 14:57:28,459 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-07 14:57:41,735 - BERTopic - Dimensionality - Completed ✓
2026-04-07 14:57:41,740 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-07 14:57:41,795 - BERTopic - Cluster - Completed ✓
2026-04-07 14:57:41,820 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-07 14:57:41,975 - BERTopic - Representation - Completed ✓


# Step 7: Results


In [ ]:
topic_info = topic_model.get_topic_info()
n_topics = len(topic_info) - 1
n_outliers = sum(1 for t in topics if t == -1)

print(f"\n{'='*50}")
print("RESULTS")
print(f"{'='*50}")
print(f"Countries analyzed: {len(documents)}")
print(f"Topics discovered: {n_topics}")
print(f"Outliers (unique positions): {n_outliers}")


RESULTS
Countries analyzed: 163
Topics discovered: 14
Outliers (unique positions): 46


# Step 8: Display topics found


In [ ]:
print(f"\n{'='*50}")
print("TOPICS DISCOVERED IN COUNTRY STATEMENTS")
print(f"{'='*50}")

for topic_id in range(-1, min(n_topics + 1, 10)):
    if topic_id == -1:
        continue  # Skip outlier topic for main display
    topic_words = topic_model.get_topic(topic_id)
    if topic_words:
        words_str = ", ".join([f"{word}" for word, score in topic_words[:8]])
        count = topic_info[topic_info['Topic'] == topic_id]['Count'].values[0]
        # Find countries in this topic
        topic_countries = [country_names[i] for i, t in enumerate(topics) if t == topic_id][:5]
        print(f"\n📌 TOPIC {topic_id}: ({count} countries)")
        print(f"   Keywords: {words_str}")
        print(f"   Example countries: {', '.join(topic_countries)}")


TOPICS DISCOVERED IN COUNTRY STATEMENTS

📌 TOPIC 0: (22 countries)
   Keywords: and, the, to, we, of, in, is, our
   Example countries: NGO CGIAR, IO IUCN, Denmark, Croatia, NGO FBO

📌 TOPIC 1: (18 countries)
   Keywords: the, and, to, of, we, our, for, in
   Example countries: Marshall Islands, El Salvador, Papua New Guinea, Samoa, Niue

📌 TOPIC 2: (10 countries)
   Keywords: the, to, and, of, we, is, our, this
   Example countries: Panama, Mauritania, San Marino, Saint Kits and Nevis, IO Colition for Rainforest-Nations

📌 TOPIC 3: (9 countries)
   Keywords: the, to, we, and, of, in, with, our
   Example countries: Portugal, UNFCCC Executive Secretary, Chile, NGO Parls, Venezuela

📌 TOPIC 4: (8 countries)
   Keywords: the, and, to, of, in, dams, climate, hydropower
   Example countries: Rwanda, IO IGAD, Zambia, NGO Waterkeepr-20Alliance, Djibouti

📌 TOPIC 5: (8 countries)
   Keywords: we, to, the, is, that, in, are, of
   Example countries: Ireland, IO ITF, Latvia, North Macedonia, S

# Step 9: Map to SDGs


In [ ]:
print(f"\n{'='*50}")
print("MAPPING TOPICS TO SUSTAINABLE DEVELOPMENT GOALS")
print(f"{'='*50}")

sdg_interpretation = {
    "emissions, coal, fossil, renewable, energy": "SDG 7 (Clean Energy) + SDG 13 (Climate Action)",
    "adaptation, resilience, vulnerable, loss, damage": "SDG 13 (Climate Action) + SDG 1 (No Poverty)",
    "finance, funding, billion, support, developing": "SDG 17 (Partnerships) + SDG 1 (No Poverty)",
    "forest, deforestation, nature, biodiversity, land": "SDG 15 (Life on Land) + SDG 13 (Climate Action)",
    "technology, innovation, transfer, capacity": "SDG 9 (Industry & Innovation) + SDG 17 (Partnerships)",
    "transparency, reporting, review, stocktake": "SDG 13 (Climate Action) + SDG 16 (Institutions)",
    "gender, women, youth, inclusion, indigenous": "SDG 5 (Gender Equality) + SDG 10 (Reduced Inequalities)"
}

print("\nBased on the keywords above, the topics align with:")
for keywords, sdgs in sdg_interpretation.items():
    print(f"  • {sdgs}")


MAPPING TOPICS TO SUSTAINABLE DEVELOPMENT GOALS

Based on the keywords above, the topics align with:
  • SDG 7 (Clean Energy) + SDG 13 (Climate Action)
  • SDG 13 (Climate Action) + SDG 1 (No Poverty)
  • SDG 17 (Partnerships) + SDG 1 (No Poverty)
  • SDG 15 (Life on Land) + SDG 13 (Climate Action)
  • SDG 9 (Industry & Innovation) + SDG 17 (Partnerships)
  • SDG 13 (Climate Action) + SDG 16 (Institutions)
  • SDG 5 (Gender Equality) + SDG 10 (Reduced Inequalities)


# Step 10: Export results


In [ ]:
results_df = pd.DataFrame({
    'country': country_names,
    'topic': topics,
    'topic_size': [topic_info[topic_info['Topic'] == t]['Count'].values[0] if t != -1 else 0 for t in topics],
    'document_length': file_sizes
})

results_df.to_csv('cop26_country_topics.csv', index=False)
print(f"\nResults saved to 'cop26_country_topics.csv'")


Results saved to 'cop26_country_topics.csv'


# Step 11: Visualizations


In [ ]:
fig1 = topic_model.visualize_topics()
fig1.show()

fig2 = topic_model.visualize_barchart(top_n_topics=min(10, n_topics))
fig2.show()

# Step 12: Download results


In [ ]:
from google.colab import files
files.download('cop26_country_topics.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>